# Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

1.Tracking agent behavior with logging, analytics, and debugging.

2.Transforming prompts, tool selection, and output formatting.

3.Adding retries, fallbacks, and early termination logic.

4.Applying rate limits, guardrails, and PII detection.

In [47]:

import os
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from IPython.display import display, Markdown

model = ChatNVIDIA(
  base_url="https://integrate.api.nvidia.com/v1",
  model="nvidia/nemotron-3.5-super-vl-120b-a12b",
  api_key=os.environ["NVIDIA_API_KEY"],
  temperature=1,
  top_p=0.95,
  max_completion_tokens=16384,
  seed=42,
)

try:
  response= model.invoke("My name is rudra , So tell me what is my name")
  print("Model in working perfectly")
  print(display(response.content))
except Exception as e:
    print("Error type:", type(e).__name__)
    print("Error details:", str(e))

Model in working perfectly


'Your name is **Rudra**. You told me at the beginning of your message!'

None


# Summarization Middleware

### Summarization Middleware
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing
older context. Summarization is useful for the following:

- Long-running conversations that exceed contextwindows.

- Multi-turn dialogues with extensive history.
Applications where preserving full conversation context matters.

In [48]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage,SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
###Message based summarization

agent= create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10),
            keep=("messages",4)
        )
        ]
)


In [49]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [50]:
# Alternative test data
question=[
    "What is the 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?"
]

for q in question:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Message: {response}")
    print(f"Message: {len(response['messages'])}")


Message: {'messages': [HumanMessage(content='What is the 2+2?', additional_kwargs={}, response_metadata={}, id='366bea58-8349-4691-b920-aa87f842bd73'), AIMessage(content='2 + 2 = 4', additional_kwargs={'reasoning_content': 'The user is asking a very simple arithmetic question: "What is the 2+2?" The answer is 4. I should provide a straightforward, concise answer. There\'s no trick here, no need for overcomplication. Just state that 2 + 2 = 4.', '_reasoning_api_fields': ['reasoning_content']}, response_metadata={'content': '2 + 2 = 4', 'role': 'assistant', 'reasoning_content': 'The user is asking a very simple arithmetic question: "What is the 2+2?" The answer is 4. I should provide a straightforward, concise answer. There\'s no trick here, no need for overcomplication. Just state that 2 + 2 = 4.', 'token_usage': {'prompt_tokens': 24, 'completion_tokens': 68, 'total_tokens': 92, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'completion_tokens_details': {'accepted_

# Token size

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver


@tool
def search_hotels (city: str) -> str:
    """Search hotels returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel 5 star, $350/night, spa, pool, gym
    2. City Inn 4 star, $180/night, business center
    3. Budget Stay 3 star, $75/night, free wifi"""


agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",550),
            keep=("tokens",200)
            )
    ]
)

config = {"configurable":{ "thread_id": "test-1"}}

def count_tokens (messages):
         total_chars = sum(len(str(m.content)) for m in messages)
         return total_chars // 4  #4 chars = 1 token


In [53]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")

Paris: ~140 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='e617bd9b-7249-4ab7-bc9b-8692f24184f6'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking to find hotels in Paris. I should use the search_hotels function with the city parameter set to "Paris".', '_reasoning_api_fields': ['reasoning_content'], 'tool_calls': [{'id': 'call-115cabb3-eff2-4708-a43e-aee57e436751', 'type': 'function', 'function': {'name': 'search_hotels', 'arguments': '{"city":"Paris"}'}}]}, response_metadata={'content': None, 'tool_calls': [{'id': 'call-115cabb3-eff2-4708-a43e-aee57e436751', 'type': 'function', 'function': {'name': 'search_hotels', 'arguments': '{"city":"Paris"}'}}], 'role': 'assistant', 'reasoning_content': 'The user is asking to find hotels in Paris. I should use the search_hotels function with the city parameter set to "Paris".', 'token_usage': {'prompt_tokens': 276, 'completion_tokens': 56, 'total_

### Human in the Loop Middleware

Pause agent execution for human approval, editing, or rejection of tool calls before they execute.
 Human-in-the-loop is useful for the following:

• High-stakes operations requiring human approval (e.g. database writes, financial transactions).

• Compliance workflows where human oversight is mandatory.

• Long-running conversations where human feedback guides the agent.


In [57]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

In [60]:
agent=create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,

            }
        )
    ]
)

In [61]:
config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [56]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='520ba310-29ca-4cd2-b283-eed8ba8a091a'),
  AIMessage(content="I'm sorry, but I don't have the capability to send emails. You can send an email to john@test.com using your own email client or service (such as Gmail, Outlook, etc.) with the subject 'Hello' and body 'How are you?'. Let me know if you need help with anything else!", additional_kwargs={'reasoning_content': "The user is asking to send an email. I need to check if I have a tool available for sending emails. Looking at the available tools, I only have `search_hotels` which is for searching hotels in a city. I do not have any email sending functionality.\n\nTherefore, I cannot fulfill the request to send an email. I should inform the user that I don't have the capability to send emails and suggest they use an email client or service directly.\n\nI will not call any tools s

In [62]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: Email sent successfully to john@test.com with subject 'Hello'.


# Reject

In [67]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model=model,
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [68]:
config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config)

In [69]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='eb7e6229-1121-4632-862d-27fa9a2df44c'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to send an email to "john@test.com" with subject "Hello" and body "How are you?".\nI have the `send_email_tool` available.\nI will call `send_email_tool` with:\nrecipient: "john@test.com"\nsubject: "Hello"\nbody: "How are you?"', '_reasoning_api_fields': ['reasoning_content'], 'tool_calls': [{'id': 'call-9fe9feba-11d2-4c4f-8d00-32b01ec11548', 'type': 'function', 'function': {'name': 'send_email_tool', 'arguments': '{"recipient":"john@test.com","subject":"Hello","body":"How are you?"}'}}]}, response_metadata={'content': None, 'tool_calls': [{'id': 'call-9fe9feba-11d2-4c4f-8d00-32b01ec11548', 'type': 'function', 'function': {'name': 'send_email_tool', 'arguments': '{"recipient":"john@test.com","subject":"Hello","b

In [73]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")